# Mortgage Delinquency Risk Preprocessing

This notebook prepares the monthly Fannie Mae loan panel for modeling. The target is `future_serious_dq_6m`, which marks whether a loan becomes 90 or more days delinquent in the next six months.

The main goal here is not to train models yet. I want to create a clean development dataset that can be used consistently in the modeling notebook. That means dropping rows without a usable target, separating features from the target, using a chronological train/test split, applying one-hot encoding to categorical fields, scaling numeric fields, and saving the processed outputs.

## 1. Imports and settings

I am starting from the processed panel created in data wrangling and reviewed in EDA. The raw Fannie Mae files are not used in this notebook because the monthly loan-level panel has already been built.

In [1]:
from pathlib import Path
import inspect
import json

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import save_npz
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
RANDOM_STATE = 42
DATASET_TAG = "2021q1_q2_sample10pct"
TARGET = "future_serious_dq_6m"
TEST_SIZE_MONTH_SHARE = 0.20
RARE_LABEL = "Other"
MISSING_LABEL = "Missing"
TOP_CATEGORY_LIMITS = {
    "seller_name": 20,
    "servicer_name": 20
}

In [3]:
def find_project_root(start_path=None):
    start_path = Path.cwd() if start_path is None else Path(start_path)
    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate
    return start_path


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROCESSED_DIR / f"preprocessing_outputs_{DATASET_TAG}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)
print("Output folder:", OUTPUT_DIR)


Project root: C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk
Processed data folder: C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\processed
Output folder: C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\processed\preprocessing_outputs_2021q1_q2_sample10pct


## 2. Load the monthly loan panel

The notebook looks for the full parquet panel first. If a sample parquet or CSV version is present instead, it can use that too. For the final project run, the full processed panel should be used.

In [4]:
panel_path = PROCESSED_DIR / f"loan_month_panel_{DATASET_TAG}.parquet"
csv_panel_path = PROCESSED_DIR / f"loan_month_panel_{DATASET_TAG}.csv"

if panel_path.exists():
    panel = pd.read_parquet(panel_path)
    loaded_path = panel_path
elif csv_panel_path.exists():
    panel = pd.read_csv(csv_panel_path)
    loaded_path = csv_panel_path
else:
    raise FileNotFoundError("No processed loan month panel was found in data/processed.")

print("Loaded:", loaded_path)
print("Shape:", panel.shape)
display(panel.head())


Loaded: C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\processed\loan_month_panel_2021q1_q2_sample10pct.parquet
Shape: (13915701, 68)


,reference_pool_id,loan_identifier,monthly_reporting_period,channel,seller_name,servicer_name,original_interest_rate,current_interest_rate,original_upb,upb_at_issuance,...,dq_status_worsened,upb_ratio,paydown_pct,future_serious_dq_6m,reporting_year,reporting_month,reporting_quarter,origination_year,origination_quarter,loan_month_key
0,NaN,000122128334,2021-01-01,R,"loanDepot.com, LLC","loanDepot.com, LLC",2.375,2.375,766000.0,NaN,...,0,1.000000,0.000000,0.0,2021,1,2021Q1,2020,2020Q4,000122128334_2021_01
1,NaN,000122128334,2021-02-01,R,"loanDepot.com, LLC","loanDepot.com, LLC",2.375,2.375,766000.0,NaN,...,0,0.997389,0.002611,0.0,2021,2,2021Q1,2020,2020Q4,000122128334_2021_02
2,NaN,000122128334,2021-03-01,R,"loanDepot.com, LLC","loanDepot.com, LLC",2.375,2.375,766000.0,NaN,...,0,0.996084,0.003916,0.0,2021,3,2021Q1,2020,2020Q4,000122128334_2021_03
3,NaN,000122128334,2021-04-01,R,"loanDepot.com, LLC","loanDepot.com, LLC",2.375,2.375,766000.0,NaN,...,0,0.993473,0.006527,0.0,2021,4,2021Q2,2020,2020Q4,000122128334_2021_04
4,NaN,000122128334,2021-05-01,R,"loanDepot.com, LLC","loanDepot.com, LLC",2.375,2.375,766000.0,NaN,...,0,0.992167,0.007833,0.0,2021,5,2021Q2,2020,2020Q4,000122128334_2021_05


## 3. Check the target before cleaning

Before dropping anything, I want to see how the target is distributed. This matters because the positive class is rare. I am not resampling in this notebook because I want the preprocessing outputs to reflect the actual portfolio distribution. Class imbalance will be handled during modeling with model settings and evaluation metrics.

In [5]:
target_counts = panel[TARGET].value_counts(dropna=False).rename_axis(TARGET).reset_index(name="row_count")
target_counts["pct"] = target_counts["row_count"] / len(panel) * 100
display(target_counts)

,future_serious_dq_6m,row_count,pct
0,0.0,13577132,97.567000
1,NaN,305482,2.195233
2,1.0,33087,0.237767


## 4. Drop rows without a usable target

Rows with a missing `future_serious_dq_6m` label do not have enough forward history to know whether the loan became seriously delinquent in the next six months. I am removing those rows from the training dataset instead of filling the target.

While reviewing the EDA, I also noticed that the target rate drops sharply in the last few reporting months of the panel. Those months can still receive a label from the wrangling step, but the label only sees part of the six-month window, so loans that go seriously delinquent right after the panel ends are counted as 0. To keep the labels honest, I am also dropping reporting months that do not have a full six months of panel data after them. This matters here because the chronological test split uses the latest months, which is exactly where those partial labels live.

In [6]:
model_panel = panel.dropna(subset=[TARGET]).copy()
model_panel[TARGET] = model_panel[TARGET].astype(int)
model_panel["monthly_reporting_period"] = pd.to_datetime(model_panel["monthly_reporting_period"])

# The 6-month target needs a full forward window. The last months of the panel only see
# part of that window, so their labels undercount delinquency. Drop them.
last_panel_month = pd.to_datetime(panel["monthly_reporting_period"]).max()
label_cutoff_month = last_panel_month - pd.DateOffset(months=6)
censored_rows = int((model_panel["monthly_reporting_period"] > label_cutoff_month).sum())
model_panel = model_panel[model_panel["monthly_reporting_period"] <= label_cutoff_month]

model_panel = model_panel.sort_values(["monthly_reporting_period", "loan_identifier"]).reset_index(drop=True)

print("Rows before target cleanup:", len(panel))
print("Last panel month:", last_panel_month.date())
print("Label cutoff month (last month with a full 6-month window):", label_cutoff_month.date())
print("Censored tail rows dropped:", f"{censored_rows:,}")
print("Rows after target cleanup:", len(model_panel))
print("Target rate after cleanup:", model_panel[TARGET].mean())

Rows before target cleanup: 13915701
Last panel month: 2025-12-01
Label cutoff month (last month with a full 6-month window): 2025-06-01
Censored tail rows dropped: 1,085,280
Rows after target cleanup: 12524939
Target rate after cleanup: 0.002490391370369149


In [7]:
monthly_target_summary = (
    model_panel
    .groupby("monthly_reporting_period")[TARGET]
    .agg(rows="size", positives="sum", target_rate="mean")
    .reset_index()
)

display(monthly_target_summary.head())
display(monthly_target_summary.tail())

,monthly_reporting_period,rows,positives,target_rate
0,2021-01-01,45009,99,0.002200
1,2021-02-01,87045,185,0.002125
2,2021-03-01,141055,286,0.002028
3,2021-04-01,190540,360,0.001889
4,2021-05-01,232977,453,0.001944


,monthly_reporting_period,rows,positives,target_rate
49,2025-02-01,223591,612,0.002737
50,2025-03-01,222788,651,0.002922
51,2025-04-01,221856,662,0.002984
52,2025-05-01,220875,680,0.003079
53,2025-06-01,219867,698,0.003175


## 5. Create a chronological train/test split

This is a forecasting-style problem, so I am not using a random split. A random split would allow later months and earlier months from the same overall period to mix together, which is less realistic for this use case. Instead, the model will train on earlier reporting months and test on later reporting months.

In [8]:
months = np.array(sorted(model_panel["monthly_reporting_period"].dropna().unique()))
split_index = int(len(months) * (1 - TEST_SIZE_MONTH_SHARE))
split_month = months[split_index]

train_mask = model_panel["monthly_reporting_period"] < split_month
test_mask = model_panel["monthly_reporting_period"] >= split_month

train_df = model_panel.loc[train_mask].copy()
test_df = model_panel.loc[test_mask].copy()

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "start_month": [train_df["monthly_reporting_period"].min(), test_df["monthly_reporting_period"].min()],
    "end_month": [train_df["monthly_reporting_period"].max(), test_df["monthly_reporting_period"].max()],
    "rows": [len(train_df), len(test_df)],
    "positive_rows": [train_df[TARGET].sum(), test_df[TARGET].sum()],
    "target_rate": [train_df[TARGET].mean(), test_df[TARGET].mean()]
})

display(split_summary)

,split,start_month,end_month,rows,positive_rows,target_rate
0,train,2021-01-01,2024-07-01,10059227,23807,0.002367
1,test,2024-08-01,2025-06-01,2465712,7385,0.002995


## 6. Select modeling features

The feature set below follows the EDA notes. I am keeping loan terms, borrower and property attributes, current monthly balance information, and delinquency behavior features that are known at the reporting month.

I am dropping identifiers, raw dates, raw loan payment history, ZIP code, MSA, and unmapped fields. Seller and servicer are kept, but they are grouped before encoding so they do not create a very wide dummy-variable matrix.

In [9]:
numeric_features = [
    "original_interest_rate",
    "current_interest_rate",
    "original_upb",
    "current_actual_upb",
    "original_loan_term",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "remaining_months_to_maturity",
    "original_ltv",
    "original_cltv",
    "number_of_borrowers",
    "dti",
    "borrower_credit_score_at_origination",
    "co_borrower_credit_score_at_origination",
    "number_of_units",
    "mortgage_insurance_percentage",
    "total_principal_current",
    "current_dq_num",
    "is_current",
    "is_30plus_dq",
    "is_60plus_dq",
    "is_90plus_dq",
    "prior_dq_num",
    "dq_status_change",
    "dq_status_worsened",
    "upb_ratio",
    "paydown_pct",
    "count_30plus_dq_past_12m",
    "count_60plus_dq_past_12m",
    "max_dq_past_12m",
    "reporting_year",
    "reporting_month",
    "origination_year"
]

categorical_features = [
    "channel",
    "seller_name",
    "servicer_name",
    "first_time_home_buyer_indicator",
    "loan_purpose",
    "property_type",
    "occupancy_status",
    "property_state",
    "amortization_type",
    "prepayment_penalty_indicator",
    "interest_only_loan_indicator",
    "modification_flag",
    "reporting_quarter",
    "origination_quarter"
]

excluded_features = [
    "reference_pool_id",
    "loan_identifier",
    "monthly_reporting_period",
    "origination_date",
    "first_payment_date",
    "maturity_date",
    "current_loan_delinquency_status",
    "loan_payment_history",
    "msa",
    "zip_code_short",
    "loan_month_key",
    "source_file",
    TARGET
]

numeric_features = [feature for feature in numeric_features if feature in model_panel.columns]
categorical_features = [feature for feature in categorical_features if feature in model_panel.columns]
model_features = numeric_features + categorical_features

feature_plan = pd.DataFrame({
    "feature": model_features,
    "preprocessing_role": ["numeric_scaled" for feature in numeric_features] + ["categorical_one_hot" for feature in categorical_features]
})

display(feature_plan)

,feature,preprocessing_role
0,original_interest_rate,numeric_scaled
1,current_interest_rate,numeric_scaled
2,original_upb,numeric_scaled
3,current_actual_upb,numeric_scaled
4,original_loan_term,numeric_scaled
5,loan_age,numeric_scaled
6,remaining_months_to_legal_maturity,numeric_scaled
7,remaining_months_to_maturity,numeric_scaled
8,original_ltv,numeric_scaled
9,original_cltv,numeric_scaled


## 7. Review missingness and cardinality before encoding

This check is mainly here to make sure the feature set is reasonable before fitting the transformer. Missing numeric values will be filled with the training median. Missing categorical values will become their own category. Categorical variables will then be one-hot encoded.

In [10]:
missing_feature_check = pd.DataFrame({
    "feature": model_features,
    "train_missing_pct": [train_df[feature].isna().mean() * 100 for feature in model_features],
    "test_missing_pct": [test_df[feature].isna().mean() * 100 for feature in model_features],
    "train_unique_values": [train_df[feature].nunique(dropna=True) for feature in model_features],
    "test_unique_values": [test_df[feature].nunique(dropna=True) for feature in model_features]
})

display(missing_feature_check.sort_values("train_missing_pct", ascending=False).head(25))

,feature,train_missing_pct,test_missing_pct,train_unique_values,test_unique_values
15,mortgage_insurance_percentage,81.221698,81.392271,17,15
13,co_borrower_credit_score_at_origination,52.722381,52.423843,216,216
22,dq_status_change,2.713827,0.000000,32,30
21,prior_dq_num,2.713827,0.000000,30,29
16,total_principal_current,2.713827,0.000000,407502,278141
7,remaining_months_to_maturity,0.049934,0.255464,360,325
12,borrower_credit_score_at_origination,0.036365,0.036785,216,216
11,dti,0.002833,0.002758,51,51
33,seller_name,0.000119,0.000000,32,26
34,servicer_name,0.000070,0.000000,43,26


## 8. Group high-cardinality seller and servicer values

Seller and servicer may contain useful portfolio information, but unrestricted one-hot encoding can create too many sparse columns. I am only keeping the most common values from the training set and grouping the rest into `Other`. This is fit on the training set only so the test set does not influence preprocessing choices.

In [11]:
def build_top_category_maps(df, columns, limits):
    maps = {}
    for column in columns:
        limit = limits.get(column)
        if limit is None:
            continue
        top_values = (
            df[column]
            .astype("string")  # leave category dtype before fillna, which rejects unseen labels
            .fillna(MISSING_LABEL)
            .astype(str)
            .value_counts()
            .head(limit)
            .index
            .tolist()
        )
        maps[column] = top_values
    return maps


def apply_top_category_maps(df, maps):
    df = df.copy()
    for column, allowed_values in maps.items():
        values = df[column].astype("string").fillna(MISSING_LABEL).astype(str)
        df[column] = values.where(values.isin(allowed_values), RARE_LABEL)
    return df


category_maps = build_top_category_maps(train_df, categorical_features, TOP_CATEGORY_LIMITS)
train_df_grouped = apply_top_category_maps(train_df, category_maps)
test_df_grouped = apply_top_category_maps(test_df, category_maps)

category_grouping_summary = pd.DataFrame({
    "feature": list(category_maps.keys()),
    "kept_categories": [len(values) for values in category_maps.values()]
})

display(category_grouping_summary)

,feature,kept_categories
0,seller_name,20
1,servicer_name,20


## 9. Separate features and target

At this point, the split has already been made and rare category handling has been fit on the training data. Now I can create the actual `X` and `y` objects for preprocessing and modeling.

In [12]:
X_train = train_df_grouped[model_features].copy()
X_test = test_df_grouped[model_features].copy()
y_train = train_df_grouped[TARGET].copy()
y_test = test_df_grouped[TARGET].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train target rate:", y_train.mean())
print("y_test target rate:", y_test.mean())

X_train shape: (10059227, 46)
X_test shape: (2465712, 46)
y_train target rate: 0.002366682847499117
y_test target rate: 0.0029950780950897754


## 10. Build the preprocessing transformer

The transformer applies median imputation and scaling to numeric variables. It applies missing-value filling and one-hot encoding to categorical variables. The transformer is fit only on the training data and then reused on the test data.

In [13]:
encoder_kwargs = {"handle_unknown": "ignore"}
if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
    encoder_kwargs["sparse_output"] = True
else:
    encoder_kwargs["sparse"] = True

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=MISSING_LABEL)),
    ("onehot", OneHotEncoder(**encoder_kwargs))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop",
    sparse_threshold=1.0
)

## 11. Fit on training data and transform both splits

This is the main preprocessing step. The important part is that the transformer learns medians, scaling values, and dummy-variable categories from the training set only.

In [14]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (10059227, 172)
Processed testing shape: (2465712, 172)


In [15]:
feature_names = preprocessor.get_feature_names_out()
feature_names = pd.Series(feature_names).str.replace("num__", "", regex=False).str.replace("cat__", "", regex=False)
feature_names = pd.DataFrame({"feature_name": feature_names})

print("Number of processed features:", len(feature_names))
display(feature_names.head(20))

Number of processed features: 172


,feature_name
0,original_interest_rate
1,current_interest_rate
2,original_upb
3,current_actual_upb
4,original_loan_term
5,loan_age
6,remaining_months_to_legal_maturity
7,remaining_months_to_maturity
8,original_ltv
9,original_cltv


## 12. Save preprocessing outputs for modeling

The modeling notebook will load these files directly. Saving the fitted preprocessing object also makes the modeling workflow reproducible because the same transformations can be applied later without refitting them on test data.

In [16]:
preprocessing_summary = pd.DataFrame({
    "item": [
        "source_rows",
        "labeled_rows",
        "train_rows",
        "test_rows",
        "train_start_month",
        "train_end_month",
        "test_start_month",
        "test_end_month",
        "target",
        "train_target_rate",
        "test_target_rate",
        "numeric_features",
        "categorical_features",
        "processed_features"
    ],
    "value": [
        len(panel),
        len(model_panel),
        len(train_df),
        len(test_df),
        str(train_df["monthly_reporting_period"].min().date()),
        str(train_df["monthly_reporting_period"].max().date()),
        str(test_df["monthly_reporting_period"].min().date()),
        str(test_df["monthly_reporting_period"].max().date()),
        TARGET,
        y_train.mean(),
        y_test.mean(),
        len(numeric_features),
        len(categorical_features),
        len(feature_names)
    ]
})

display(preprocessing_summary)

,item,value
0,source_rows,13915701
1,labeled_rows,12524939
2,train_rows,10059227
3,test_rows,2465712
4,train_start_month,2021-01-01
5,train_end_month,2024-07-01
6,test_start_month,2024-08-01
7,test_end_month,2025-06-01
8,target,future_serious_dq_6m
9,train_target_rate,0.002367


In [17]:
save_npz(OUTPUT_DIR / "X_train_processed.npz", X_train_processed)
save_npz(OUTPUT_DIR / "X_test_processed.npz", X_test_processed)
y_train.to_frame(TARGET).to_csv(OUTPUT_DIR / "y_train.csv", index=False)
y_test.to_frame(TARGET).to_csv(OUTPUT_DIR / "y_test.csv", index=False)
test_df_grouped[["loan_identifier", "monthly_reporting_period"]].to_csv(OUTPUT_DIR / "test_meta.csv", index=False)
feature_names.to_csv(OUTPUT_DIR / "feature_names.csv", index=False)
preprocessing_summary.to_csv(OUTPUT_DIR / "preprocessing_summary.csv", index=False)
feature_plan.to_csv(OUTPUT_DIR / "feature_plan.csv", index=False)
missing_feature_check.to_csv(OUTPUT_DIR / "missing_feature_check.csv", index=False)
split_summary.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)
category_grouping_summary.to_csv(OUTPUT_DIR / "category_grouping_summary.csv", index=False)
with open(OUTPUT_DIR / "category_maps.json", "w") as f:
    json.dump(category_maps, f, indent=2)
joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")

print("Saved preprocessing outputs to:", OUTPUT_DIR)

Saved preprocessing outputs to: C:\Users\Esh\OneDrive\Documents\Mortgage Delinquency Forecasting Risk\data\processed\preprocessing_outputs_2021q1_q2_sample10pct


## 13. Preprocessing checkpoint

The preprocessing stage is now ready for modeling. The output keeps the target definition from the earlier notebooks, uses a chronological split, preserves known monthly delinquency behavior, avoids unrestricted high-cardinality one-hot encoding, standardizes numeric magnitudes, and saves the fitted transformer plus the processed train/test matrices.

The target is still highly imbalanced, so the modeling notebook should not rely on accuracy alone. I will compare models with metrics that fit rare-event classification, especially precision, recall, F1, ROC-AUC, PR-AUC, and confusion matrices at different thresholds.